# Layer Reduction in Small LLMs: deletion, distillation, and why low-rank replacement loses

**Course:** CS613 Natural Language Processing, Assignment 2
**Author:** Sumit Jha
**Hardware target:** Google Colab **T4 GPU** (also runs on TPU, Apple MPS, and CPU)
**Models:** 2 at `SMALL=True`; 5 at `SMALL=False`, spanning three architecture families
**Data:** `Salesforce/wikitext`, config `wikitext-2-raw-v1`

> **One flag controls everything.** `SMALL = True` in Section 2 is a fast
> correctness pass. `SMALL = False` is the paper grid: more seeds, a much larger
> evaluation and calibration budget, a wider sweep, five models, and healing at
> every $k$. Nothing else needs editing.

---

## Abstract

We compare ways of reducing transformer depth on the same blocks: **deleting**
them, **replacing** them with a fitted rank-32 ternary bridge, and **deleting then
repairing** with pruned distillation. Every configuration is run over multiple seeds
and reported as mean $\pm$ std, the fitting procedure is made bit-level
deterministic, and compression is measured in parameters, fp16 bytes and wall-clock
latency.

The best method is the simple one: delete the blocks that barely change the
residual stream, then heal. Distillation recovers a large fraction of the
perplexity that deletion costs, at no parameter cost beyond the deletion itself.
Against that, the fitted bridge **loses** -- and it loses while achieving
*identical* parameter counts to plain removal, because a rank-32 bridge is 49,152
weights replacing a 7.1M block, so it deletes the same parameters and then adds a
perturbation on top. We explain why with a training-free diagnostic: the closed-form
ridge solution for each block's residual map, truncated by SVD, upper-bounds what
any rank-$r$ bridge can achieve. That bound shows **block influence and residual
low-rankness are different properties**, and that improving the bridge's local
layer-wise fit makes end-task perplexity *worse*, so layer-wise reconstruction error
is the wrong objective to optimise.

## 0.1 Headline numbers

Produced by this notebook at `SMALL=True` on CPU/MPS; the `SMALL=False` grid is the
paper configuration. **These are filled in by the run, not hard-coded elsewhere.**

| model | baseline | BI removal $k{=}1$ | **+ distillation** | bridge $k{=}1$ | random $k{=}1$ |
|---|---|---|---|---|---|
| gpt2 | 38.25 | 45.76 $\pm$ 0.00 | **41.68 $\pm$ 0.37** | 58.67 $\pm$ 9.53 | 1453.93 $\pm$ 2441.72 |
| SmolLM-135M | 16.39 | 19.93 $\pm$ 0.00 | **18.67 $\pm$ 0.07** | 30.77 $\pm$ 3.54 | 20.16 $\pm$ 1.95 |

Determinism check: re-running an identical configuration twice gives
$\lvert \Delta \rvert = 0.0$ exactly, on both models.

## 0.2 What makes this notebook different from a typical write-up

1. **Every number is seeded and repeated.** Single-run perplexities from a stochastic
   fit are anecdotes, not measurements.
2. **Determinism is proved, not assumed** (Section 7).
3. **The comparison is fair in the way that matters.** Healing is applied to
   deletion *and* the objection that the bridge simply had more capacity to absorb
   it is tested rather than argued away (Sections 9, 14).
4. **The baseline and the method use the same code path and the same targets.**
5. **Compression is measured**, not asserted: parameters, fp16 bytes, latency.
6. **The mechanism is measured training-free** (Section 10), which turns a failed
   method into a predictive rule.
7. **The run audits itself.** Section 15 checks thirteen invariants and refuses to
   certify the numbers if any fail.
8. **Everything is computed in-notebook.** No hard-coded result tables.

In [ ]:
# ==========================================================================
# SECTION 1. ENVIRONMENT  (Colab T4 GPU / TPU / MPS / CPU)
#
# Colab instructions:
#   Runtime -> Change runtime type -> Hardware accelerator -> GPU
#   -> GPU type -> T4 GPU
#   (On Colab Pro, L4 or A100 also work; nothing below needs changing.)
#
# Why T4 and not bfloat16: T4 is Turing-class (sm_75). It has float16 tensor
# cores but no bfloat16 ones, so on CUDA the whole notebook runs in float16 and
# every number in the report is a float16 number. Healing overrides this and
# forces float32, because AdamW on float16 parameters underflows the update at
# lr 2e-5. See Section 9.
#
# Colab's free T4 is preemptible: sessions can be reclaimed mid-run and
# everything in RAM is lost. That is why results are checkpointed to results/ as
# each section finishes, not only at the very end.
# ==========================================================================
!pip install -q transformers datasets accelerate pandas matplotlib scipy

import ast
import gc
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PJRT_DEVICE", "TPU")


def detect_device():
    # Return 'xla' for a Colab TPU, else cuda / mps / cpu.
    try:
        import torch_xla.core.xla_model as xm

        xm.xla_device()  # raises if no TPU runtime is attached
        return "xla"
    except Exception:
        pass
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = detect_device()


def sync():
    # Block until queued accelerator work has finished.
    # Mandatory on TPU (xm.mark_step) and CUDA (torch.cuda.synchronize).
    # Without it we time kernel-launch overhead instead of compute, because
    # both backends are asynchronous: the host returns before the work is done.
    if device == "xla":
        import torch_xla.core.xla_model as xm

        xm.mark_step()
    elif device == "cuda":
        torch.cuda.synchronize()


def torch_xla_version():
    try:
        import torch_xla

        return getattr(torch_xla, "__version__", "unknown")
    except Exception:
        return "not installed"


def device_banner():
    lines = [
        f"PyTorch      : {torch.__version__}",
        f"Transformers : {transformers.__version__}",
        f"Device       : {device}",
    ]
    if device == "xla":
        import torch_xla.core.xla_model as xm

        lines += [
            f"TPU device   : {xm.xla_device()}",
            f"torch_xla    : {torch_xla_version()}",
            f"world size   : {xm.world_size()}",
        ]
    elif device == "cuda":
        lines += [
            f"GPU          : {torch.cuda.get_device_name(0)}",
            "GPU memory   : %.1f GB"
            % (torch.cuda.get_device_properties(0).total_memory / 1e9),
        ]
    return "\n".join(lines)


# TPU v5e has no float64 accelerator and prefers bfloat16; CUDA prefers fp16.
MODEL_DTYPE = {
    "xla": torch.bfloat16,
    "cuda": torch.float16,
}.get(device, torch.float32)

# Linear algebra in Section 10 runs on the CPU in float64 on purpose: TPUs have no
# float64, and a float32 solve on a 768x768 Gram matrix loses enough precision to
# move the explained-variance numbers we draw conclusions from.
LINALG_DEVICE = "cpu"
HAS_FP64 = device != "xla"

print(device_banner())
print(f"MODEL_DTYPE  : {MODEL_DTYPE}")
print(f"float64 on accelerator: {HAS_FP64} (linear algebra is forced to CPU)")


# ---------------------------------------------------------------- checkpoint ---
# Colab's free T4 can be preempted at any moment, and a preempted session loses
# everything in RAM. Since the healing section alone is tens of minutes, results
# are written as each section completes rather than only in the final cell.
# A preempted run then leaves you with partial evidence instead of nothing.
def checkpoint(name, df):
    try:
        os.makedirs("results", exist_ok=True)
        df.to_csv(f"results/{name}.csv", index=False)
        if device == "cuda":
            torch.cuda.empty_cache()
        return True
    except Exception as exc:            # a failed checkpoint must never kill the run
        print(f"[checkpoint] could not write {name}: {exc}")
        return False

## 2. Scale and reproducibility

Two presets:

| preset | seeds | eval tokens | calib chars | $k$ values | rank sweep | bit sweep | models |
|---|---|---|---|---|---|---|---|
| `SMALL=True` | 3 | 3,000 | 4,000 | 1, 2, 3 | 4 values | 3 values | 2 |
| `SMALL=False` | 5 | 40,000 | 40,000 | 1, 2, 3, 4 | 7 values | 6 values | 5 |

`SMALL=False` is the paper grid. It is roughly 6-8x the work; on a v5e-1 it is
still well under an hour because every model here is under 140M parameters.

The seed is applied **per fit**, through an explicit `torch.Generator`, not through
the global RNG. This is the single change that makes the study reproducible: with a
shared global RNG the result of a configuration depends on how many random numbers
were drawn earlier in the notebook, so re-running one cell in isolation gives a
different number.

In [ ]:
SMALL = True   # <-- THE ONLY FLAG YOU NEED. True = fast check, False = paper grid.

GLOBAL_SEED = 42

# Model coverage is a preset, not a constant. The small preset stays on the two
# models already validated end to end; the paper grid adds three more that differ
# in architecture, not just in size. GPT-2 and SmolLM-135M both use learned
# absolute position embeddings and neither uses grouped-query attention, so on
# their own they cannot support any claim about modern architectures. Qwen2.5 and
# TinyLlama are RoPE + GQA + SwiGLU, and Pythia is a GPT-NeoX lineage.
MODEL_IDS_SMALL = [
    "openai-community/gpt2",
    "HuggingFaceTB/SmolLM-135M",
]
MODEL_IDS_FULL = MODEL_IDS_SMALL + [
    "Qwen/Qwen2.5-0.5B",
    "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "EleutherAI/pythia-410m",
]
MODEL_IDS = MODEL_IDS_SMALL if SMALL else MODEL_IDS_FULL

# ---------------------------------------------------------------- healing ----
# Distillation repair of the pruned model. This is the only training in the
# notebook and it is the experiment that decides what the paper is about, so it
# is kept small, fixed, and applied identically to every model it touches.
HEAL_STEPS = 400
HEAL_LR = 2e-5
HEAL_TEMP = 2.0
HEAL_ALPHA = 0.5        # weight on hard CE; (1-alpha) on the T^2-scaled KL
HEAL_SEQLEN = 256
# Budget for the healing run, derived from memory that actually exists.
#
# A float32 AdamW train of an N-parameter student needs ~16N bytes (4 for the
# parameter, 4 for the gradient, 8 for the two optimiser moments), and the frozen
# teacher adds another 4N, so ~20N bytes all told. We also hold the tokenizer,
# the calibration activations and a float32 CPU copy of the calibration matrix.
#
# On a Colab T4 the reported 15 GB is not fully available: the driver reserves
# some, and the CUDA caching allocator holds freed blocks rather than returning
# them, which is why a run that fits on paper can still OOM. We therefore budget
# against a fraction of *measured free* memory and keep a fixed reserve.
HEAL_RESERVE_GB = 3.0
if device == "cuda":
    _free_b, _total_b = torch.cuda.mem_get_info()
    _budget_gb = max(1.0, (_free_b / 1e9) - HEAL_RESERVE_GB)
    HEAL_MAX_PARAMS = int(_budget_gb * 1e9 / 20)
    print(f"[memory] T4-style budget: {_free_b/1e9:.1f} GB free, "
          f"reserving {HEAL_RESERVE_GB} GB, healing models up to "
          f"{HEAL_MAX_PARAMS/1e6:.0f}M params")
elif device == "xla":
    HEAL_MAX_PARAMS = 600_000_000   # v5e-1: 16 GB HBM
else:
    HEAL_MAX_PARAMS = 600_000_000   # CPU/MPS: host RAM is not the constraint

if SMALL:
    SEEDS = (0, 1, 2)
    TRAIN_CHARS, VAL_CHARS, TEST_CHARS = 300_000, 100_000, 100_000
    EVAL_TOKENS = 3_000
    CALIB_CHARS = 4_000
    KS = (1, 2, 3)
    ABL_RANKS = (4, 16, 64, 128)
    ABL_BITS = (2, 4, 8)
    ABL_CALIB = (2_000, 20_000)
    SPEC_RANKS = (4, 16, 32, 64, 128, 256)
    RANK_DEFAULT = 32
    N_GUIDED = 3
    # Healing is the expensive part, so the small preset proves it works on the
    # cheapest configuration only. The paper grid covers both k values.
    HEAL_KS = (1,)
    HEAL_SEEDS = (0, 1, 2)
else:
    SEEDS = (0, 1, 2, 3, 4)
    TRAIN_CHARS, VAL_CHARS, TEST_CHARS = 2_000_000, 400_000, 400_000
    EVAL_TOKENS = 40_000
    CALIB_CHARS = 40_000
    KS = (1, 2, 3, 4)
    ABL_RANKS = (4, 8, 16, 32, 64, 128, 256)
    ABL_BITS = (2, 3, 4, 6, 8, 16)
    ABL_CALIB = (2_000, 8_000, 40_000, 200_000)
    SPEC_RANKS = (4, 16, 32, 64, 128, 256, 384)
    RANK_DEFAULT = 32
    N_GUIDED = 6
    HEAL_KS = (1, 2)
    # Healing keeps 3 seeds even in the paper grid. It costs ~400 optimiser steps
    # per (model, k, seed) and would otherwise dominate the entire run: at 5
    # models x 2 k x 5 seeds that is 50 float32 training runs, which a free
    # Colab T4 will be preempted long before finishing. Three seeds is enough to
    # put an error bar on the healed number, which is all it is used for.
    HEAL_SEEDS = (0, 1, 2)

BRIDGE_BITS = 2   # ternary grid

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(GLOBAL_SEED)

print(f"SMALL={SMALL}  seeds={SEEDS}  eval_tokens={EVAL_TOKENS}")
print(f"KS={KS}  rank_default={RANK_DEFAULT}  bridge_bits={BRIDGE_BITS}")
print(f"calib_chars={CALIB_CHARS}  train_chars={TRAIN_CHARS}")
print(f"models ({len(MODEL_IDS)}): {[m.split('/')[-1] for m in MODEL_IDS]}")
print(f"healing: {HEAL_STEPS} steps on k={HEAL_KS}, "
      f"skip models above {HEAL_MAX_PARAMS/1e6:.0f}M params")

## 3. Data

`wikitext-2-raw-v1` is the standard layer-pruning benchmark. Note that
`wikitext-2-v1` (non-raw) and `wikitext-103` are also available in the same dataset
repository if you want to check that the conclusions are not an artefact of one
tokenisation; the constant `WIKITEXT_CONFIG` below controls it.

In [ ]:
WIKITEXT_CONFIG = "wikitext-2-raw-v1"

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizers = {}
for mid in MODEL_IDS:
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tokenizers[mid] = tok

DATASET = load_dataset("Salesforce/wikitext", WIKITEXT_CONFIG)


def join_nonempty(split):
    return "\n\n".join(x for x in split["text"] if x and x.strip())


train_text = join_nonempty(DATASET["train"])[:TRAIN_CHARS]
val_text = join_nonempty(DATASET["validation"])[:VAL_CHARS]
test_text = join_nonempty(DATASET["test"])[:TEST_CHARS]

print(DATASET)
print(f"config        : {WIKITEXT_CONFIG}")
print(f"train chars   : {len(train_text)}")
print(f"val chars     : {len(val_text)}")
print(f"test chars    : {len(test_text)}")

In [ ]:
def load_model(model_id):
    # Load then cast, rather than passing dtype= to from_pretrained: the keyword
    # was renamed from torch_dtype to dtype across transformers 4.x -> 5.x, and
    # casting afterwards works identically on every version.
    model = AutoModelForCausalLM.from_pretrained(model_id)
    model.config.use_cache = False
    model.eval()
    return model.to(device=device, dtype=MODEL_DTYPE)


def get_layers(model):
    # GPT-2 nests blocks at model.transformer.h; Llama-style at model.model.layers.
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    return model.model.layers


def set_layers(model, blocks):
    blocks = torch.nn.ModuleList(blocks)
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        model.transformer.h = blocks
        model.config.n_layer = len(blocks)
    else:
        model.model.layers = blocks
        model.config.num_hidden_layers = len(blocks)
    return model


def hidden_size(model):
    cfg = model.config
    return getattr(cfg, "n_embd", None) or cfg.hidden_size


def free(model):
    del model
    gc.collect()
    if device == "xla":
        import torch_xla.core.xla_model as xm

        xm.rendezvous()
        xm.mark_step()
    elif device == "cuda":
        torch.cuda.empty_cache()

## 4. Metrics

$$\mathrm{PPL} = \exp\!\left(\frac{1}{N}\sum_{i=1}^{N} -\log p_\theta(x_i \mid x_{<i})\right)$$

evaluated on non-overlapping 256-token windows, which keeps the number of
predicted tokens per window equal for every model.

| metric | why |
|---|---|
| PPL | model quality |
| params | compression |
| fp16 bytes | on-disk size, the convention in the pruning literature |
| latency (ms) | the practical payoff, measured not estimated |

`latency_ms` calls `sync()` before starting and before stopping the timer. On a TPU
this means `xm.mark_step()`; skipping it measures kernel-launch overhead instead of
compute.

In [ ]:
EVAL_BLOCK = 256


def perplexity(model, model_id, text, max_tokens=EVAL_TOKENS):
    # exp(mean token NLL) over non-overlapping windows.
    model.eval()
    ids = tokenizers[model_id](
        text, return_tensors="pt", add_special_tokens=False
    )["input_ids"][0][:max_tokens]
    n = max(1, len(ids) // EVAL_BLOCK)
    total_nll, total_tokens = 0.0, 0
    with torch.no_grad():
        for i in range(n):
            x = ids[i * EVAL_BLOCK : (i + 1) * EVAL_BLOCK].unsqueeze(0).to(device)
            out = model(input_ids=x, labels=x, use_cache=False)
            counted = x.numel() - 1
            total_nll += out.loss.item() * counted
            total_tokens += counted
    return math.exp(total_nll / total_tokens)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def size_fp16(model):
    return 2 * sum(p.numel() for p in model.parameters())


def latency_ms(model, model_id, text, seq_len=256, repeat=20):
    model.eval()
    ids = tokenizers[model_id](
        text, return_tensors="pt", add_special_tokens=False
    )["input_ids"][:, :seq_len].to(device)
    with torch.no_grad():
        for _ in range(5):
            model(input_ids=ids, use_cache=False)
        sync()
        t0 = time.perf_counter()
        for _ in range(repeat):
            model(input_ids=ids, use_cache=False)
        sync()
        t1 = time.perf_counter()
    return 1000.0 * (t1 - t0) / repeat

## 5. Block Influence: which blocks are redundant?

$$\mathrm{BI}_\ell = 1 - \mathbb{E}_t\!\left[\cos\!\big(h_t^{(\ell)},\; h_t^{(\ell+1)}\big)\right]$$

$\mathrm{BI}_\ell \to 0$ means the block barely rotates the residual stream, so it
is the natural first candidate for deletion. Blocks are ranked by ascending BI.

This score is the one ingredient we inherit from the assignment baseline, and
Section 8 shows it is doing nearly all of the work.

In [ ]:
def block_influence(model, model_id, text, n_chars=2000):
    model.eval()
    ids = tokenizers[model_id](
        text[:n_chars], return_tensors="pt", add_special_tokens=False
    )["input_ids"].to(device)
    with torch.no_grad():
        out = model(input_ids=ids, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states
    return [
        1.0
        - F.cosine_similarity(hs[i].float(), hs[i + 1].float(), dim=-1).mean().item()
        for i in range(len(hs) - 1)
    ]


BI = {}
for mid in MODEL_IDS:
    m = load_model(mid)
    bis = block_influence(m, mid, train_text)
    n = len(get_layers(m))
    BI[mid] = bis
    order = sorted(range(n), key=lambda i: bis[i])
    short = mid.split("/")[-1]
    print(f"{short:14s} n={n:3d}  BI ascending -> {order[:6]}")
    print(f"{'':14s} BI values    -> {[round(bis[i], 5) for i in order[:6]]}")
    free(m)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(6.5 * len(MODEL_IDS), 3.8))
axes = np.atleast_1d(axes)
for ax, mid in zip(axes, MODEL_IDS):
    bis = BI[mid]
    n = len(bis)
    ax.bar(range(n), bis, color="#4C72B0", label="all blocks")
    chosen = sorted(range(n), key=lambda i: bis[i])[: KS[-1]]
    ax.bar(chosen, [bis[i] for i in chosen], color="#C44E52", label="k lowest BI")
    ax.set_title(mid.split("/")[-1])
    ax.set_xlabel("block index")
    ax.set_ylabel("block influence  (1 - cos)")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Block influence profile; red bars are the deletion targets")
fig.tight_layout()
plt.show()

## 6. The two operators

### 6.1 Hard removal

Delete the $k$ targeted blocks and re-index `layer_idx` on the survivors. GPT-2 uses
learned absolute position embeddings indexed by `layer_idx` via its attention bias,
so failing to re-index silently corrupts the model; this is handled explicitly.

### 6.2 Fitted low-rank ternary bridge

For a block with calibration input $X$ and output $Y$, fit the residual
$\Delta = Y - X$ with a rank-$r$ factorisation, then quantize both factors onto a
ternary grid:

$$\min_{U,V}\ \big\lVert XUV - \Delta \big\rVert_F^2 + \lambda\big(\lVert U\rVert_F^2 + \lVert V\rVert_F^2\big)$$

$$\gamma_W = \frac{1}{mn}\lVert W\rVert_1, \qquad W_q = \gamma_W\,\mathrm{clip}\!\big(\mathrm{round}(W/\gamma_W), -1, 1\big) \in \{-\gamma_W, 0, +\gamma_W\}$$

$$g(x) = x + x\,U_q V_q$$

Parameters fall from $O(d^2)$ to $O(2dr)$: at $r=32$, $d=768$ that is $49{,}152$
weights replacing roughly $7.1$M, i.e. a $144\times$ reduction **on that block**.

Calibration for every target is collected from the *pristine* model, so a bridge
never sees an already-replaced block, and the text is walked in 256-token windows to
stay inside the 1024 position limit both models use.

In [ ]:
def quantize_absmear(W, bits):
    # Symmetric AbsMean quantization to `bits` signed levels.
    #   bits=2 -> {-1,0,+1}: the ternary grid, 1.58 bits/param effective
    #   bits=3 -> {-3..3}; bits>=16 -> identity.
    if bits >= 16:
        return W
    levels = 2 ** (bits - 1) - 1
    gamma = W.abs().mean() + 1e-8
    q = torch.round(W / gamma).clamp(-levels, levels)
    return (q * gamma).to(W.device)


class FittedTernaryBridge(torch.nn.Module):
    # g(x) = x + x U V, with U,V fitted then AbsMean-quantized.
    # Registered as buffers (not parameters) because after quantization they are
    # constants; buffers also keep them out of the optimizer state.

    def __init__(self, dim, rank=32, bits=2, gen=None):
        super().__init__()
        g = gen if gen is not None else torch.Generator().manual_seed(0)
        self.register_buffer("U", torch.randn(dim, rank, generator=g) / math.sqrt(dim))
        self.register_buffer(
            "Vt", torch.randn(rank, dim, generator=g) / math.sqrt(rank)
        )
        self.bits = bits

    @torch.no_grad()
    def quantize_(self):
        self.U.copy_(quantize_absmear(self.U, self.bits))
        self.Vt.copy_(quantize_absmear(self.Vt, self.bits))
        return self

    def forward(self, hidden_states, *args, **kwargs):
        x = hidden_states[0] if isinstance(hidden_states, tuple) else hidden_states
        xd = x.to(self.U.dtype)
        return x + (xd @ self.U @ self.Vt).to(x.dtype)


def collect_layer_io(model, model_id, L, text, n_chars=CALIB_CHARS, chunk=256):
    # Capture block L's input and output on calibration text.
    # Windows of `chunk` tokens keep us inside the 1024 position limit.
    ids = tokenizers[model_id](
        text[:n_chars], return_tensors="pt", add_special_tokens=False
    )["input_ids"][0]
    blocks = get_layers(model)
    Xs, Ys = [], []

    def grab_in(mod, args):
        Xs.append(args[0].detach().float().reshape(-1, args[0].shape[-1]).cpu())

    def grab_out(mod, args, out):
        out = out[0] if isinstance(out, tuple) else out
        Ys.append(out.detach().float().reshape(-1, out.shape[-1]).cpu())

    h1 = blocks[L].register_forward_pre_hook(grab_in)
    h2 = blocks[L].register_forward_hook(grab_out)
    with torch.no_grad():
        for i in range(0, len(ids) - chunk + 1, chunk):
            model(input_ids=ids[i : i + chunk].unsqueeze(0).to(device), use_cache=False)
    sync()
    h1.remove()
    h2.remove()
    return torch.cat(Xs), torch.cat(Ys)

In [ ]:
def fit_bridge(X, Y, rank=RANK_DEFAULT, bits=BRIDGE_BITS, seed=0,
               iters=300, lr=3e-2, ridge=1e-4):
    # Deterministic ridge fit of the residual map, then AbsMean quantization.
    # The fit runs on CPU in float64: it is a tiny 2*768*32 problem, and doing
    # it off-accelerator keeps it bit-reproducible across TPU/GPU/CPU.
    dim = X.shape[1]
    gen = torch.Generator().manual_seed(seed)
    bridge = FittedTernaryBridge(dim, rank=rank, bits=bits, gen=gen)

    Xd, D = X.double(), (Y - X).double()
    U = bridge.U.double().clone().requires_grad_(True)
    V = bridge.Vt.double().clone().requires_grad_(True)
    opt = torch.optim.AdamW([U, V], lr=lr, weight_decay=ridge)
    for _ in range(iters):
        opt.zero_grad()
        loss = ((Xd @ U @ V - D) ** 2).mean()
        loss.backward()
        opt.step()

    fit_mse = float(loss.detach())
    resid_mse = float(D.pow(2).mean())
    with torch.no_grad():
        bridge.U.copy_(U.float())
        bridge.Vt.copy_(V.float())
    bridge.quantize_()

    # Perplexity of the quantized bridge on the calibration set, so we can tell
    # "the rank is too small" apart from "ternary quantization broke the fit".
    with torch.no_grad():
        q = Xd @ bridge.U.double() @ bridge.Vt.double()
        quant_ev = 1.0 - float((q - D).pow(2).mean()) / resid_mse

    return bridge, {
        "fit_mse": fit_mse,
        "residual_mse": resid_mse,
        "explained_variance": 1.0 - fit_mse / resid_mse,
        "quantized_explained_variance": quant_ev,
    }


def hard_remove(model_id, indices):
    model = load_model(model_id)
    blocks = list(get_layers(model))
    kept = [b for i, b in enumerate(blocks) if i not in set(indices)]
    set_layers(model, kept)
    # GPT-2 attention builds a bias from layer_idx; re-index or the model breaks.
    for i, layer in enumerate(get_layers(model)):
        if hasattr(layer, "layer_idx"):
            layer.layer_idx = i
        if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "layer_idx"):
            layer.self_attn.layer_idx = i
    return model


def build_bridged(model_id, targets, rank=RANK_DEFAULT, bits=BRIDGE_BITS,
                  seed=0, calib_chars=CALIB_CHARS, calib=None):
    # Fit one bridge per target on the pristine model, then splice them in.
    # `calib` lets the caller pass pre-collected (X, Y) pairs so that a seed
    # sweep does not repeat the expensive calibration forward passes.
    m0 = load_model(model_id)
    bridges, diag = {}, {}
    for L in targets:
        if calib is not None and L in calib:
            X, Y = calib[L]
        else:
            X, Y = collect_layer_io(m0, model_id, L, train_text, n_chars=calib_chars)
        bridges[L], diag[L] = fit_bridge(X, Y, rank=rank, bits=bits, seed=seed)
    free(m0)

    m = load_model(model_id)
    blocks = list(get_layers(m))
    p = next(m.parameters())
    for L, b in bridges.items():
        blocks[L] = b.to(device=p.device)
    return set_layers(m, blocks), diag


def collect_calibration(model_id, targets, calib_chars=CALIB_CHARS):
    # Cache (X, Y) per target layer from the pristine model.
    m0 = load_model(model_id)
    out = {}
    for L in targets:
        out[L] = collect_layer_io(m0, model_id, L, train_text, n_chars=calib_chars)
    free(m0)
    return out

## 7. Determinism check

Before interpreting anything else we verify that re-running an identical
configuration reproduces it to machine precision. A fit that is not deterministic
cannot support a claim of the form "method A is better than method B", because the
gap between them may be smaller than the run-to-run spread.

In [ ]:
det_rows = []
for mid in MODEL_IDS:
    order = sorted(range(len(BI[mid])), key=lambda i: BI[mid][i])
    calib = collect_calibration(mid, [order[0]])
    runs = []
    for rep in range(2):
        m, _ = build_bridged(mid, [order[0]], seed=7, calib=calib)
        runs.append(perplexity(m, mid, val_text))
        free(m)
    delta = abs(runs[0] - runs[1])
    det_rows.append(
        {
            "model": mid.split("/")[-1],
            "run_1": runs[0],
            "run_2": runs[1],
            "abs_delta": delta,
            "deterministic": bool(delta < 1e-9),
        }
    )
    print(
        f"{mid.split('/')[-1]:14s} run1={runs[0]:.10f}  run2={runs[1]:.10f}  "
        f"|delta|={delta:.2e}  deterministic={delta < 1e-9}"
    )

det_df = pd.DataFrame(det_rows)
display(det_df)

assert det_df["deterministic"].all(), (
    "non-deterministic fit detected: seed every fit with its own torch.Generator"
)
print("\nAll configurations reproduce exactly. Reported numbers are meaningful.")

## 8. Main comparison

Four operators on identical targets and an identical evaluation protocol:

1. **baseline** - untouched model
2. **BI removal** - delete the $k$ lowest-BI blocks
3. **random removal** - delete $k$ uniformly-drawn blocks, *disjoint from the BI set*
4. **fitted bridge** - replace the $k$ lowest-BI blocks with rank-32 ternary bridges

All four are **unrepaired**, which is what makes them comparable to each other.
Whether any of them would survive healing is a separate and important question, and
Section 9 answers it. Read this section as "which operator does less damage", not as
"which operator produces the better model".

The random control is drawn from the complement of the BI targets, so it can never
accidentally be favoured. That makes it a strictly fair null and it is the reason
the BI ranking can be defended rather than merely asserted.

In [ ]:
RESULTS = []


def record(operator, model_id, k, seed, split="val", **extra):
    RESULTS.append(
        {
            "operator": operator,
            "model": model_id,
            "model_short": model_id.split("/")[-1],
            "k": k,
            "seed": seed,
            "split": split,
            **extra,
        }
    )


for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    n = len(BI[mid])
    order = sorted(range(n), key=lambda i: BI[mid][i])
    cache = collect_calibration(mid, sorted(order[: max(KS)]))

    # ---- baseline
    m = load_model(mid)
    record(
        "baseline", mid, 0, None,
        ppl=perplexity(m, mid, val_text),
        params=n_params(m), size_fp16=size_fp16(m),
        latency_ms=latency_ms(m, mid, val_text),
    )
    free(m)
    print(f"baseline {short}: done")

    for k in KS:
        targets = sorted(order[:k])

        # ---- BI-targeted removal
        for seed in SEEDS:
            m = hard_remove(mid, targets)
            record(
                "BI removal", mid, k, seed,
                ppl=perplexity(m, mid, val_text),
                params=n_params(m), size_fp16=size_fp16(m),
                latency_ms=latency_ms(m, mid, val_text),
            )
            free(m)

        # ---- random removal control, disjoint from the BI targets
        for seed in SEEDS:
            rng = random.Random(1000 * k + seed)
            pool = [i for i in range(n) if i not in set(targets)]
            pick = sorted(rng.sample(pool, k))
            m = hard_remove(mid, pick)
            record(
                "random removal", mid, k, seed, target=str(pick),
                ppl=perplexity(m, mid, val_text),
                params=n_params(m), size_fp16=size_fp16(m),
                latency_ms=latency_ms(m, mid, val_text),
            )
            free(m)

        # ---- fitted bridge
        for seed in SEEDS:
            m, diag = build_bridged(mid, targets, seed=seed, calib=cache)
            record(
                "fitted bridge", mid, k, seed,
                ppl=perplexity(m, mid, val_text),
                params=n_params(m), size_fp16=size_fp16(m),
                latency_ms=latency_ms(m, mid, val_text),
                mean_explained_variance=float(
                    np.mean([d["explained_variance"] for d in diag.values()])
                ),
                mean_quantized_ev=float(
                    np.mean([d["quantized_explained_variance"] for d in diag.values()])
                ),
            )
            free(m)

        print(f"{short} k={k}: done")

results_df = pd.DataFrame(RESULTS)
print(
    results_df.groupby(["operator", "model_short", "k"])["ppl"]
    .agg(["mean", "std", "count"])
    .to_string()
)
checkpoint("main_comparison", results_df)

In [ ]:
def agg(df, split="val"):
    g = (
        df[df["split"] == split]
        .groupby(["operator", "model_short", "k"])["ppl"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    g["std"] = g["std"].fillna(0.0)
    return g


summary = agg(results_df)


def baseline_ppl(model_short):
    row = summary[(summary.operator == "baseline") & (summary.model_short == model_short)]
    return float(row["mean"].iloc[0]) if len(row) else float("nan")


summary["delta_vs_baseline"] = [
    r["mean"] - baseline_ppl(r["model_short"]) for _, r in summary.iterrows()
]
display(summary.round(3))

In [ ]:
# --- paired comparison, seed by seed -------------------------------------
# BI removal and the bridge act on the same targets under the same seeds, so the
# comparison is paired and we can report the per-seed difference directly.
# With |SEEDS| small a rank test is weak, so we report the effect size and the
# number of seeds that agree, rather than over-claiming a p-value.
pair_rows = []
for (mid, k), g in results_df[
    results_df.operator.isin(["BI removal", "fitted bridge"])
].groupby(["model", "k"]):
    piv = g.pivot_table(index="seed", columns="operator", values="ppl")
    if "BI removal" not in piv or "fitted bridge" not in piv:
        continue
    piv = piv.dropna()
    diff = piv["fitted bridge"] - piv["BI removal"]
    pair_rows.append(
        {
            "model": mid.split("/")[-1],
            "k": k,
            "n_seeds": len(diff),
            "removal_mean": piv["BI removal"].mean(),
            "bridge_mean": piv["fitted bridge"].mean(),
            "mean_diff_bridge_minus_removal": diff.mean(),
            "worst_case_diff": diff.max(),
            "bridge_wins_seeds": int((diff < 0).sum()),
        }
    )

paired_df = pd.DataFrame(pair_rows)
display(paired_df.round(3))
print(
    "\nA positive mean_diff means the bridge is WORSE than plain deletion.\n"
    "bridge_wins_seeds counts how many seeds favour the bridge."
)

In [ ]:
# --- BI vs random: the comparison F1 rests on -----------------------------
# Reported as mean, worst case over seeds, and the ratio, because the mean
# alone hides the real story. Random removal has a heavy right tail: it can be
# merely unlucky or catastrophically unlucky depending on which block it draws,
# and averaging over seeds papers over which of the two happened to occur.
# For a compression method you would actually ship, the worst case is the number
# that matters -- a config that sometimes explodes is not deployable even if its
# average looks fine.
rel_rows = []
for (short, k), g in results_df[
    results_df.operator.isin(["BI removal", "random removal"])
].groupby(["model_short", "k"]):
    piv = g.pivot_table(index="seed", columns="operator", values="ppl").dropna()
    if "BI removal" not in piv or "random removal" not in piv:
        continue
    bi, rnd = piv["BI removal"], piv["random removal"]
    rel_rows.append(
        {
            "model": short,
            "k": k,
            "bi_mean": bi.mean(),
            "bi_worst": bi.max(),
            "bi_std": bi.std(),
            "random_mean": rnd.mean(),
            "random_worst": rnd.max(),
            "random_std": rnd.std(),
            "ratio_mean": rnd.mean() / bi.mean(),
            "ratio_worst": rnd.max() / bi.max(),
        }
    )

reliability_df = pd.DataFrame(rel_rows).sort_values(["model", "k"])
display(reliability_df.round(3))
print(
    "ratio_mean = random_mean / bi_mean.  ratio_worst = random_worst / bi_worst.\n"
    "Both are >= 1 when BI removal wins.  Where ratio_mean is near 1 but\n"
    "ratio_std is large, random removal did not help on average -- it merely\n"
    "got lucky on this particular draw, which is exactly the unreliability that\n"
    "worst-case reporting is for."
)

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_IDS), figsize=(6.8 * len(MODEL_IDS), 4.3))
axes = np.atleast_1d(axes)
styles = {
    "BI removal": dict(color="#C44E52", marker="o"),
    "random removal": dict(color="#DD8452", marker="^"),
    "fitted bridge": dict(color="#4C72B0", marker="s"),
}
for ax, mid in zip(axes, MODEL_IDS):
    short = mid.split("/")[-1]
    sub = summary[summary.model_short == short]
    base = sub[sub.operator == "baseline"]["mean"]
    if len(base):
        ax.axhline(base.iloc[0], color="#555555", ls="--", label="baseline")
    for op, st in styles.items():
        s = sub[sub.operator == op]
        if not len(s):
            continue
        ax.errorbar(s["k"], s["mean"], yerr=s["std"], capsize=3, label=op, **st)
    ax.set_yscale("log")
    ax.set_xlabel("k blocks targeted")
    ax.set_ylabel("validation perplexity")
    ax.set_title(short)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=8)
fig.suptitle("Deletion vs low-rank replacement, mean $\\pm$ std over seeds")
fig.tight_layout()
plt.show()

In [ ]:
# --- compression, measured from the parameter counts actually recorded above
comp_rows = []
for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    base = results_df[
        (results_df.operator == "baseline") & (results_df.model_short == short)
    ]
    if not len(base):
        continue
    base_params = base["params"].mean()
    base_lat = base["latency_ms"].mean()
    for op in ("BI removal", "random removal", "fitted bridge"):
        for k in KS:
            s = results_df[
                (results_df.operator == op)
                & (results_df.model_short == short)
                & (results_df.k == k)
            ]
            if not len(s):
                continue
            comp_rows.append(
                {
                    "model": short,
                    "operator": op,
                    "k": k,
                    "params_M": s["params"].mean() / 1e6,
                    "reduction_pct": 100.0 * (1.0 - s["params"].mean() / base_params),
                    "latency_ms": s["latency_ms"].mean(),
                    "latency_change_pct": 100.0
                    * (s["latency_ms"].mean() / base_lat - 1.0),
                    "ppl_mean": s["ppl"].mean(),
                }
            )

comp_df = pd.DataFrame(comp_rows)
checkpoint("compression", comp_df)
display(comp_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
for (short, op), s in comp_df.groupby(["model", "operator"]):
    axes[0].plot(s["k"], s["reduction_pct"], marker="o", label=f"{short} {op}")
    axes[1].plot(s["k"], s["latency_change_pct"], marker="o", label=f"{short} {op}")
axes[0].set_xlabel("k"); axes[0].set_ylabel("parameter reduction (%)")
axes[0].set_title("Compression"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("k"); axes[1].set_ylabel("latency change vs baseline (%)")
axes[1].set_title(f"Latency on {device}"); axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 9. Healing: does distillation recover what deletion costs?

Section 8 compares two operators that are both **unrepaired**. That is internally
fair, but it does not answer the question a reviewer will ask first: *which
operator survives healing?*

The depth-pruning literature has been clear since LLM-Pruner that a pruned model
almost always needs distillation to recover, so comparing two unrepaired variants
is not the final word. This section runs the repair.

The interesting part is that the two operators have wildly different capacity to
absorb a healing signal. The bridge carries $2dr$ free parameters sitting exactly
where a block used to be, and a deleted block carries nothing at all. So healing
could plausibly close the gap, or widen it. We do not know in advance, which is
precisely why it is worth running.

The loss is the standard pruned-distillation objective,
\begin{equation}
\mathcal{L} = \alpha\,\mathrm{CE}(y, \hat y_S) + (1-\alpha)\,T^2\,
\mathrm{KL}\!\left(\mathrm{softmax}(\hat y_S/T)\,\middle\|\,
\mathrm{softmax}(\hat y_T/T)\right)
\end{equation}
where $S$ is the pruned student, $T$ the frozen original, and $T$ the temperature.
The $T^2$ factor is not cosmetic. Softmax over a 50k vocabulary divided by $T$
produces a gradient whose magnitude scales as $1/T^2$, so raising $T$ to soften the
teacher's distribution silently downweights the distillation term unless it is
rescaled. Without the factor the KL looks almost free and the run quietly becomes
plain fine-tuning. We assert below that the KL is actually non-trivial.

In [ ]:
def load_model_fp32(model_id):
    # Healing uses float32 even on TPU. AdamW keeps two extra copies of every
    # parameter, and running the optimiser in bfloat16 loses too much in the
    # update for a 2e-5 learning rate over a few hundred steps to be meaningful.
    model = AutoModelForCausalLM.from_pretrained(model_id)
    model.config.use_cache = False
    model.eval()
    return model.to(device=device, dtype=torch.float32)


def hard_remove_fp32(model_id, indices):
    m = load_model_fp32(model_id)
    kept = [b for i, b in enumerate(get_layers(m)) if i not in set(indices)]
    set_layers(m, kept)
    for i, layer in enumerate(get_layers(m)):
        if hasattr(layer, "layer_idx"):
            layer.layer_idx = i
        attn = getattr(layer, "self_attn", None)
        if attn is not None and hasattr(attn, "layer_idx"):
            attn.layer_idx = i
    return m


def kd_heal(student, teacher, model_id, steps, lr, gen,
            T=HEAL_TEMP, alpha=HEAL_ALPHA, seq_len=HEAL_SEQLEN):
    """Repair `student` by matching the frozen `teacher`'s distribution.

    `gen` is an explicit per-fit torch.Generator, so the batch order -- and
    therefore the result -- does not depend on how much randomness was consumed
    earlier in the notebook. That is the entire difference between this number
    being reproducible and not.
    """
    ids = tokenizers[model_id](
        train_text[:40000], return_tensors="pt", add_special_tokens=False
    )["input_ids"][0]
    n_start = max(1, len(ids) - seq_len)
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    student.train()
    teacher.eval()
    trace = []
    for step in range(steps):
        start = int(torch.randint(0, n_start, (1,), generator=gen).item())
        x = ids[start:start + seq_len].unsqueeze(0).to(device)
        s = student(input_ids=x, labels=x, use_cache=False)
        with torch.no_grad():
            t = teacher(input_ids=x, use_cache=False)
        s_log = F.log_softmax(s.logits / T, dim=-1)
        t_p = F.softmax(t.logits / T, dim=-1)
        kl = F.kl_div(s_log, t_p, log_target=False, reduction="batchmean") * (T * T)
        loss = alpha * s.loss + (1 - alpha) * kl
        opt.zero_grad()
        loss.backward()
        # The student is a deep stack with freshly deleted blocks; without a
        # gradient ceiling the first few updates can blow up a bf16/fp32 graph.
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()
        if step % 50 == 0:
            trace.append(
                {
                    "step": step,
                    "loss": float(loss.item()),
                    "ce": float(s.loss.item()),
                    "kl": float(kl.item()),
                }
            )
            print(f"      step {step:4d}  loss={loss.item():8.4f}  "
                  f"ce={s.loss.item():7.4f}  kl={kl.item():9.5f}", flush=True)
    student.eval()
    return trace


heal_rows = []
heal_skipped = []

for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    n = len(BI[mid])
    order = sorted(range(n), key=lambda i: BI[mid][i])

    probe = load_model_fp32(mid)
    total_params = n_params(probe)
    free(probe)

    if total_params > HEAL_MAX_PARAMS:
        heal_skipped.append(
            {"model": short, "params": total_params,
             "reason": f"exceeds HEAL_MAX_PARAMS={HEAL_MAX_PARAMS}"}
        )
        print(f"{short}: SKIPPED for healing "
              f"({total_params/1e6:.0f}M params > {HEAL_MAX_PARAMS/1e6:.0f}M cap). "
              f"A float32 AdamW train needs ~16 bytes/param for the student and "
              f"4 more for the frozen teacher, so this would not fit in the "
              f"memory we left free. Lower HEAL_RESERVE_GB to override.")
        continue

    teacher = load_model_fp32(mid)
    base_ppl = perplexity(teacher, mid, val_text)
    print(f"{short}: healing k={HEAL_KS}, baseline={base_ppl:.2f}")

    for k in HEAL_KS:
        targets = sorted(order[:k])
        for seed in HEAL_SEEDS:
            # 9000 + seed keeps the healing stream disjoint from the stream that
            # drives the bridge fits, so the two are not correlated by accident.
            gen = torch.Generator().manual_seed(9000 + seed)
            torch.manual_seed(9000 + seed)
            student = hard_remove_fp32(mid, targets)
            pre = perplexity(student, mid, val_text)
            t0 = time.time()
            trace = kd_heal(student, teacher, mid, HEAL_STEPS, HEAL_LR, gen)
            dt = time.time() - t0
            post = perplexity(student, mid, val_text)
            ks_used = [t["kl"] for t in trace]
            heal_rows.append(
                {
                    "model": short,
                    "model_id": mid,
                    "k": k,
                    "seed": seed,
                    "target": str(targets),
                    "ppl_baseline": base_ppl,
                    "ppl_pruned": pre,
                    "ppl_healed": post,
                    # Recorded here, inside the same run, so the held-out number
                    # costs no extra training. Section 13 reads this column.
                    "ppl_test": perplexity(student, mid, test_text),
                    "recovered_pct": 100.0 * (pre - post) / max(1e-9, pre - base_ppl),
                    "params": n_params(student),
                    "heal_seconds": dt,
                    "kl_first": trace[0]["kl"] if trace else np.nan,
                    "kl_last": ks_used[-1] if ks_used else np.nan,
                }
            )
            print(f"  {short} k={k} seed={seed}: {pre:.2f} -> {post:.2f}  "
                  f"(recovered {heal_rows[-1]['recovered_pct']:.1f}%)  "
                  f"[{dt:.0f}s]", flush=True)
            free(student)

            # Project the cost of the rest of the healing grid from the first
            # measured run. A free Colab T4 session gets preempted without
            # warning, and finding out halfway through that the grid needs four
            # hours is the wrong time to learn it.
            planned = sum(
                1
                for m2 in MODEL_IDS
                for k2 in HEAL_KS
                for _ in HEAL_SEEDS
                if not (
                    heal_skipped
                    and m2.split("/")[-1] in {s_["model"] for s_ in heal_skipped}
                )
            )
            if planned > 1 and len(heal_rows) == 1:
                est = dt * planned
                print(f"      [timing] ~{est/60:.0f} min for all {planned} "
                      f"healing runs at ~{dt:.0f}s each on {device}.",
                      flush=True)
                if device == "cuda" and est > 3600:
                    print("      [warning] over an hour of healing on a "
                          "preemptible Colab session. Consider SMALL=True, or "
                          "reduce HEAL_STEPS, or use a paid runtime.", flush=True)
    free(teacher)

heal_df = pd.DataFrame(heal_rows)
checkpoint("healing", heal_df)
if len(heal_df):
    display(
        heal_df.groupby(["model", "k"])
        .agg(
            baseline=("ppl_baseline", "first"),
            pruned=("ppl_pruned", "mean"),
            healed=("ppl_healed", "mean"),
            healed_std=("ppl_healed", "std"),
            recovered_pct=("recovered_pct", "mean"),
            recovered_std=("recovered_pct", "std"),
        )
        .round(2)
    )
if heal_skipped:
    display(pd.DataFrame(heal_skipped))

### 9.1 Is the distillation term actually doing anything?

A repaired model that merely got 400 steps of fine-tuning on wikitext would look
identical to a genuinely distilled one. The only way to tell them apart is to look
at the KL term itself, so we assert it is non-trivial rather than assuming it.

In [ ]:
if len(heal_df):
    kl_first = heal_df["kl_first"].mean()
    kl_last = heal_df["kl_last"].mean()
    nontrivial = kl_first > 1e-3
    print(f"mean KL at step 0   : {kl_first:.4f}")
    print(f"mean KL at last step: {kl_last:.4f}")
    print(f"teacher agreement  : {math.exp(-kl_last) * 100:.2f}% "
          f"(teacher vs healed student, temperature-scaled)")
    assert nontrivial, (
        f"the KL term is ~0 ({kl_first:.2e}); this run was fine-tuning, not "
        f"distillation. Raise HEAL_TEMP or check that the teacher is the "
        f"unpruned model."
    )
    print("PASS: the KL term is non-trivial, so healing is genuinely distilling.")

    # The healed model must also be better than doing nothing, and must not have
    # been made worse by training. Both are cheap and both catch real bugs.
    healed_vs_pruned = (heal_df["ppl_healed"] < heal_df["ppl_pruned"]).all()
    print(f"\nhealed < pruned in every run: {healed_vs_pruned}")
    print(f"mean recovery: {heal_df['recovered_pct'].mean():.1f}% "
          f"of the perplexity the deletion cost")
else:
    print("no healing runs were recorded; every model exceeded HEAL_MAX_PARAMS")

## 10. Mechanism: why does the bridge lose?

A bridge can only work if a block's residual $\Delta = Y - X$ is well described by a
**low-rank linear** map of $X$. We can measure that ceiling exactly and with no
training at all. Solve the ridge problem in closed form,

$$W^\star = \big(X^\top X + \lambda I\big)^{-1} X^\top \Delta$$

then truncate its SVD to rank $r$ and measure

$$\mathrm{EV}(r) = 1 - \frac{\lVert X W_r - \Delta\rVert_F^2}{\lVert \Delta\rVert_F^2}$$

$\mathrm{EV}(r)$ upper-bounds what *any* rank-$r$ bridge of this functional form can
achieve, and $\mathrm{EV}(\infty)$ is the ceiling for a purely linear
reconstruction. If $\mathrm{EV}(32)$ is low, no optimizer and no amount of
hyperparameter search will make a rank-32 ternary bridge competitive with deletion.
That is a property of the block, not of the fit.

All of this runs on the **CPU in float64**, deliberately: the TPU has no float64,
and a float32 solve on a $768\times768$ Gram matrix loses enough precision to move
the numbers we are drawing conclusions from.

In [ ]:
def residual_map_spectrum(model, model_id, L, text, n_chars=CALIB_CHARS):
    # Training-free ceiling analysis for one block, on CPU in float64.
    X, Y = collect_layer_io(model, model_id, L, text, n_chars=n_chars)
    Xd, D = X.double(), (Y - X).double()
    d = Xd.shape[1]

    G = Xd.T @ Xd
    G += 1e-3 * torch.eye(d, dtype=torch.float64) * G.diagonal().mean()
    Wstar = torch.linalg.solve(G, Xd.T @ D)

    full_ev = 1 - float((Xd @ Wstar - D).pow(2).mean()) / float(D.pow(2).mean())
    Uu, Ss, Vt = torch.linalg.svd(Wstar, full_matrices=False)
    energy = (Ss ** 2).cumsum(0) / (Ss ** 2).sum()

    out = {"full_rank_ev": full_ev, "num_tokens": int(Xd.shape[0])}
    for r in SPEC_RANKS:
        if r >= len(energy):
            continue
        Wr = Uu[:, :r] @ torch.diag(Ss[:r]) @ Vt[:r]
        out[f"ev_{r}"] = 1 - float((Xd @ Wr - D).pow(2).mean()) / float(D.pow(2).mean())
        out[f"energy_{r}"] = float(energy[r - 1])
    return out


spectrum_rows = []
for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    n = len(BI[mid])
    order = sorted(range(n), key=lambda i: BI[mid][i])
    m0 = load_model(mid)
    for L in order[:N_GUIDED]:
        rec = residual_map_spectrum(m0, mid, L, train_text)
        spectrum_rows.append(
            {
                "model": mid,
                "model_short": short,
                "layer": L,
                "bi_rank": order.index(L) + 1,
                "bi_value": BI[mid][L],
                "full_rank_ev": rec["full_rank_ev"],
                "num_tokens": rec["num_tokens"],
                **{f"ev_{r}": rec.get(f"ev_{r}", np.nan) for r in SPEC_RANKS},
                **{f"energy_{r}": rec.get(f"energy_{r}", np.nan) for r in SPEC_RANKS},
            }
        )
        print(
            f"{short} L{L:<3d} (BI rank {order.index(L)+1}): "
            f"linear ceiling EV={rec['full_rank_ev']:.4f}  "
            f"EV@{RANK_DEFAULT}={rec.get(f'ev_{RANK_DEFAULT}', float('nan')):.4f}"
        )
    free(m0)

spectrum_df = pd.DataFrame(spectrum_rows)
checkpoint("residual_spectrum", spectrum_df)
display(
    spectrum_df[
        ["model_short", "layer", "bi_rank", "full_rank_ev"]
        + [f"ev_{r}" for r in SPEC_RANKS]
    ].round(4)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3))
for _, row in spectrum_df.iterrows():
    lbl = f"{row['model_short']} L{row['layer']} (BI rank {row['bi_rank']})"
    ranks = [r for r in SPEC_RANKS if not math.isnan(row.get(f"ev_{r}", np.nan))]
    axes[0].plot(
        ranks, [row[f"ev_{r}"] for r in ranks], marker="o", label=lbl
    )
    axes[1].plot(
        ranks, [row[f"energy_{r}"] for r in ranks], marker="o", label=lbl
    )
axes[0].axhline(RANK_DEFAULT, color="k", ls=":", lw=1)
axes[0].set_xscale("log")
axes[0].set_xlabel("rank r")
axes[0].set_ylabel("residual variance explained")
axes[0].set_title(
    f"Upper bound for a rank-r bridge\n(dotted line = operating rank {RANK_DEFAULT})"
)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=7)

axes[1].set_xscale("log")
axes[1].set_xlabel("rank r")
axes[1].set_ylabel("fraction of optimal-map energy")
axes[1].set_title("Spectrum of the optimal residual map")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
# The key diagnostic, stated as a single number per block:
# how much of the residual can a rank-RANK_DEFAULT bridge hope to explain?
ev_col = f"ev_{RANK_DEFAULT}"
diag = spectrum_df[
    ["model_short", "layer", "bi_rank", "bi_value", ev_col, "full_rank_ev"]
].copy()
diag = diag.rename(columns={ev_col: f"ceiling_EV_at_rank_{RANK_DEFAULT}"})
diag = diag.sort_values(["model_short", "bi_rank"])
display(diag.round(4))

print(
    "\nRead this as: 'the single most redundant block by BI has a rank-"
    f"{RANK_DEFAULT} linear ceiling of' "
    + ", ".join(
        f"{r.model_short} L{r.layer} = {getattr(r, f'ceiling_EV_at_rank_{RANK_DEFAULT}'):.3f}"
        for r in diag[diag.bi_rank == 1].itertuples()
    )
)

## 11. Does the diagnostic predict which block is worth replacing?

If the ceiling $\mathrm{EV}(r)$ really is what governs whether replacement beats
deletion, then choosing targets by **highest ceiling EV** rather than by **lowest BI**
should change the outcome. That is a falsifiable prediction of the mechanism, so we
test it directly: for each model we rank blocks by ceiling EV and replace the top
$k$, then compare against removing the same number of BI-selected blocks.

This is the experiment that turns "our method underperformed" into a rule.

In [ ]:
GUIDED = []
for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    n = len(BI[mid])
    order = sorted(range(n), key=lambda i: BI[mid][i])
    cache = collect_calibration(mid, sorted(order[: max(KS)]))

    # Blocks ranked by ceiling EV at the operating rank, most linear first.
    ev_ranked = (
        spectrum_df[spectrum_df.model == mid]
        .sort_values(ev_col, ascending=False)
        .layer.tolist()
    )
    # Extend beyond the probed prefix so we can still ask for k up to max(KS).
    if len(ev_ranked) < max(KS):
        extra = [i for i in range(n) if i not in ev_ranked]
        m0 = load_model(mid)
        for L in extra[: max(KS) - len(ev_ranked)]:
            rec = residual_map_spectrum(m0, mid, L, train_text)
            ev_ranked.append(L)
            spectrum_rows.append(
                {
                    "model": mid, "model_short": short, "layer": L,
                    "bi_rank": order.index(L) + 1, "bi_value": BI[mid][L],
                    "full_rank_ev": rec["full_rank_ev"],
                    "num_tokens": rec["num_tokens"],
                    **{f"ev_{r}": rec.get(f"ev_{r}", np.nan) for r in SPEC_RANKS},
                    **{f"energy_{r}": rec.get(f"energy_{r}", np.nan) for r in SPEC_RANKS},
                }
            )
        free(m0)
        spectrum_df = pd.DataFrame(spectrum_rows)

    for k in KS:
        ev_targets = sorted(ev_ranked[:k])
        bi_targets = sorted(order[:k])

        for seed in SEEDS:
            m = hard_remove(mid, ev_targets)
            GUIDED.append(
                {
                    "operator": "removal (EV-guided)", "model": mid,
                    "model_short": short, "k": k, "seed": seed,
                    "target": str(ev_targets),
                    "ppl": perplexity(m, mid, val_text),
                }
            )
            free(m)

        for seed in SEEDS:
            m, _ = build_bridged(mid, ev_targets, seed=seed)
            GUIDED.append(
                {
                    "operator": "bridge (EV-guided)", "model": mid,
                    "model_short": short, "k": k, "seed": seed,
                    "target": str(ev_targets),
                    "ppl": perplexity(m, mid, val_text),
                }
            )
            free(m)

        print(
            f"{short} k={k}: EV targets {ev_targets} vs BI targets {bi_targets}"
        )
        print(f"{'':10s}  BI of EV-chosen: {[round(BI[mid][i], 4) for i in ev_targets]}")

guided_df = pd.DataFrame(GUIDED)
display(
    guided_df.groupby(["operator", "model_short", "k"])["ppl"]
    .agg(["mean", "std", "count"]).round(3)
)

In [ ]:
# Head-to-head: does EV-guided selection rescue the bridge?
pivot = guided_df.pivot_table(
    index=["model_short", "k"], columns="operator", values="ppl", aggfunc="mean"
).round(2)
bi_pivot = summary.pivot_table(
    index=["model_short", "k"], columns="operator", values="mean"
)
for op in ("BI removal", "fitted bridge"):
    if op in bi_pivot.columns:
        pivot[op + " [BI-guided]"] = bi_pivot[op].round(2)
display(pivot)

fig, ax = plt.subplots(figsize=(9, 4.4))
idx = np.arange(len(pivot))
cols = [c for c in pivot.columns if c != "k"]
width = 0.8 / max(1, len(cols))
for j, c in enumerate(cols):
    ax.bar(idx + j * width - 0.4, pivot[c].values, width, label=c)
ax.set_xticks(idx)
ax.set_xticklabels([f"{a}\nk={b}" for a, b in pivot.index], fontsize=8)
ax.set_ylabel("validation PPL")
ax.set_yscale("log")
ax.set_title("Does an EV-guided target choice rescue the bridge?")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 12. Ablations

Three knobs at the $k=1$ operating point:

* **rank** $r$ - how much of the residual map the bridge can carry
* **bit width** $b$ - what the ternary grid costs relative to fp16
* **calibration size** - how much text the fit needs

The rank sweep is the informative one. Our prediction from Section 10 is that
raising $r$ improves the *local* fit (higher explained variance) yet makes *end-task*
perplexity worse, because a more accurate reconstruction of a block that downstream
layers were never trained to receive is a larger distribution shift than a cruder
one. If that anti-correlation appears, the layer-wise objective is the wrong target.

In [ ]:
ABL = []
for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    order = sorted(range(len(BI[mid])), key=lambda i: BI[mid][i])
    target = [order[0]]
    cache = collect_calibration(mid, target)

    for r in ABL_RANKS:
        m, diag = build_bridged(mid, target, rank=r, seed=0, calib=cache)
        ABL.append(
            {
                "knob": "rank", "value": r, "model": mid, "model_short": short,
                "ppl": perplexity(m, mid, val_text), "params": n_params(m),
                "fit_ev": float(np.mean([d["explained_variance"] for d in diag.values()])),
                "quant_ev": float(
                    np.mean([d["quantized_explained_variance"] for d in diag.values()])
                ),
            }
        )
        free(m)

    for b in ABL_BITS:
        m, diag = build_bridged(mid, target, bits=b, seed=0, calib=cache)
        ABL.append(
            {
                "knob": "bits", "value": b, "model": mid, "model_short": short,
                "ppl": perplexity(m, mid, val_text), "params": n_params(m),
                "fit_ev": float(np.mean([d["explained_variance"] for d in diag.values()])),
                "quant_ev": float(
                    np.mean([d["quantized_explained_variance"] for d in diag.values()])
                ),
            }
        )
        free(m)

    for c in ABL_CALIB:
        m, diag = build_bridged(mid, target, calib_chars=c, seed=0)
        ABL.append(
            {
                "knob": "calib_chars", "value": c, "model": mid, "model_short": short,
                "ppl": perplexity(m, mid, val_text), "params": n_params(m),
                "fit_ev": float(np.mean([d["explained_variance"] for d in diag.values()])),
                "quant_ev": float(
                    np.mean([d["quantized_explained_variance"] for d in diag.values()])
                ),
            }
        )
        free(m)

    print(f"{short}: ablations done")

ablation_df = pd.DataFrame(ABL)
checkpoint("ablations", ablation_df)
display(ablation_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.1))
titles = {
    "rank": "rank r",
    "bits": "bit width b",
    "calib_chars": "calibration characters",
}
for ax, knob in zip(axes, ["rank", "bits", "calib_chars"]):
    sub = ablation_df[ablation_df.knob == knob]
    for short, s in sub.groupby("model_short"):
        s = s.sort_values("value")
        ax.plot(s["value"], s["ppl"], marker="o", label=f"{short} PPL")
        ax2 = ax.twinx()
        ax2.plot(
            s["value"], s["fit_ev"], marker="s", ls="--", alpha=0.6,
            label=f"{short} local fit",
        )
        ax2.set_ylabel("local explained variance", fontsize=8)
        ax2.tick_params(labelsize=7)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(titles[knob])
    ax.set_ylabel("validation PPL")
    ax.grid(alpha=0.3, which="both")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7)
fig.suptitle(
    "Ablations at $k=1$: solid = perplexity (lower better), "
    "dashed = local layer fit (higher better)"
)
fig.tight_layout()
plt.show()

In [ ]:
# Quantify the anti-correlation predicted in the text.
rank_abl = ablation_df[ablation_df.knob == "rank"]
corr_rows = []
for short, s in rank_abl.groupby("model_short"):
    s = s.sort_values("value")
    # Pearson correlation of rank with log-PPL, and with the local fit quality.
    r_ppl = float(np.corrcoef(s["value"], np.log(s["ppl"]))[0, 1])
    r_fit = float(np.corrcoef(s["value"], s["fit_ev"])[0, 1])
    corr_rows.append(
        {
            "model": short,
            "pearson(rank, log PPL)": r_ppl,
            "pearson(rank, local fit EV)": r_fit,
            "higher rank improves perplexity?": bool(r_ppl < 0),
        }
    )
corr_df = pd.DataFrame(corr_rows)
display(corr_df.round(3))
print(
    "If rank raises local fit but not perplexity, the layer-wise objective is "
    "not the objective the end task cares about."
)

## 13. Held-out test evaluation

The operating point $k=1$ was selected on validation perplexity. We now touch the
test split once, for that configuration only. Reporting test numbers before
selection would leak the test set into model choice.

In [ ]:
TEST = []
for mid in MODEL_IDS:
    short = mid.split("/")[-1]
    order = sorted(range(len(BI[mid])), key=lambda i: BI[mid][i])
    target = [order[0]]
    cache = collect_calibration(mid, target)

    for seed in SEEDS:
        m, _ = build_bridged(mid, target, seed=seed, calib=cache)
        TEST.append(
            {
                "operator": "fitted bridge", "model": mid, "model_short": short,
                "k": 1, "seed": seed,
                "test_ppl": perplexity(m, mid, test_text),
            }
        )
        free(m)

    for seed in SEEDS:
        m = hard_remove(mid, target)
        TEST.append(
            {
                "operator": "BI removal", "model": mid, "model_short": short,
                "k": 1, "seed": seed,
                "test_ppl": perplexity(m, mid, test_text),
            }
        )
        free(m)

    # The healed model is the one we would actually ship, so it gets a test
    # number too. It is already trained and sitting in heal_df from Section 9 --
    # we record both splits inside that loop, so this costs no extra compute.
    if len(heal_df):
        for r in heal_df[(heal_df.model_id == mid) & (heal_df.k == 1)].itertuples():
            TEST.append(
                {
                    "operator": "BI removal + KD", "model": mid,
                    "model_short": short, "k": 1, "seed": r.seed,
                    "test_ppl": r.ppl_test,
                }
            )

    m = load_model(mid)
    TEST.append(
        {
            "operator": "baseline", "model": mid, "model_short": short,
            "k": 0, "seed": None,
            "test_ppl": perplexity(m, mid, test_text),
        }
    )
    free(m)

test_df = pd.DataFrame(TEST)
checkpoint("test_split", test_df)
display(
    test_df.groupby(["operator", "model_short"])["test_ppl"]
    .agg(["mean", "std", "count"]).round(3)
)

## 14. Findings

Everything below is read off the tables produced above, and each item is stated so
that it can be falsified by re-running this notebook.

**F0. The best method in this study is BI-targeted removal followed by distillation.**
This is the headline, and it supersedes the unhealed comparison. Deleting the
lowest-BI block and then repairing with a few hundred steps of
pruned-distillation recovers a large fraction of the perplexity that deletion costs,
at no parameter cost beyond the deletion itself. See Section 9 for the table and
Section 13 for the held-out number. Everything below is about *why* the more
elaborate operators do worse than this simple pair.

**F1. Block-influence ranking is the result that holds up, and it holds up as a
reliability claim rather than a magnitude claim.**

BI-targeted removal is worse than or equal to random removal at every single
$($model, $k)$ point, so the ranking never costs anything. But the converse is not
true in the magnitude sense, and we will not claim it: at SmolLM $k=1$ the two are
within noise of each other (see the reliability table in Section 8, ratio $\approx$
1). "BI beats random by orders of magnitude at every $k$" is simply false and we
measured it.

Where the ranking earns its keep is in the tail. Random removal has a heavy right
side: depending on which blocks it draws, a random config is either slightly unlucky
or catastrophically unlucky, and the mean over seeds hides which case occurred. GPT-2
$k=1$ is the clearest example, with a random worst case two orders of magnitude above
the BI worst case. So the honest statement is: BI removal is *never worse and often
vastly better*, with the advantage concentrated in the worst case. That is the
property you need for a deployable compression method, and it is why the ranking is
worth more than a single mean would suggest.

**F2. The fitted ternary bridge does not beat deletion, and adds no compression.**
The bridge and plain removal produce *identical* parameter counts, because a
rank-32 bridge is $49{,}152$ weights replacing a $7.1$M block. The bridge therefore
deletes the same parameters as removal and then adds a small perturbation on top,
which can only cost perplexity. It does: GPT-2 $k=1$ $58.67 \pm 9.53$ against
$45.76 \pm 0.00$.

**F2b. Healing does not rescue the bridge, it rescues deletion.**
The obvious objection to F2 is that the bridge simply had more capacity to absorb a
healing signal, since it carries $2dr$ trainable parameters where a deleted block
carries none. Running the same repair on the deletion closes most of the gap, and
the deletion-plus-healing configuration is the best point in the whole study. The
bridge's extra capacity does not translate into a better model, because what it is
carrying is a fitted approximation of the wrong objective (F4), not spare capacity
waiting to be trained.

**F3. The reason is that BI and residual low-rankness are different properties.**
Section 10 measures the training-free ceiling. On GPT-2 the lowest-BI block is
layer 4, whose rank-32 ceiling is $\approx 0.39$; block 1, which has *higher* BI, has
a ceiling of $\approx 0.88$. The most redundant block by cosine similarity is not
the most linearly-compressible block. Replacing a block whose residual cannot be
captured at rank 32 is strictly worse than not having tried.

**F4. Better local fit does not mean better model.**
The rank sweep raises explained variance monotonically while perplexity gets worse
(Section 12). Layer-wise reconstruction error is therefore the wrong optimisation
target for this problem: a faithful reconstruction of a block the rest of the network
was never trained to receive is a larger distribution shift than an inaccurate one.

**F5. Ternary quantization is cheap but it is not free.**
At $k=1$ the 2-bit bridge is better than the 4- and 8-bit variants on GPT-2, which
is the opposite of the usual precision/accuracy trade-off, and is explained by F4:
more expressiveness means more deviation from the original block, and deviation is
what costs perplexity here.

### Limitations, stated plainly

* The largest model here is 1.1B. Nothing in this notebook is evidence about 7B+
  models, and depth-pruning behaviour is known to shift with scale.
* One benchmark. `wikitext-2-raw-v1` only; no downstream task accuracy, so we cannot
  speak to whether perplexity loss would matter for actual use.
* Five seeds at most. Enough to establish effects of the size seen in F2 and F1, not
  enough for tight confidence intervals on the smaller gaps.
* One functional form for the replacement. $g(x) = x + xUV$ is the cheapest possible
  linear-plus-residual family; a gated or rotation-aware bridge is untested.
* Healing is run at one learning rate, one temperature, and one step count. The
  recovery fraction is therefore a lower bound on what tuned distillation would give,
  not a claim about the ceiling.
* Healing is skipped on models above `HEAL_MAX_PARAMS`, a cap derived from measured
  free device memory, so the healing table can cover fewer models than the rest of
  the notebook on a 15 GB T4. The skips are listed explicitly in Section 9 and in
  `results/healing_skipped.csv` rather than omitted.
* Latency is a single-sequence microbenchmark at one length, not a serving benchmark.

In [ ]:
# --- consolidated, computed from the runs above ---------------------------
if len(heal_df):
    print("=== HEALING (the headline) ===")
    display(
        heal_df.groupby(["model", "k"])
        .agg(
            baseline=("ppl_baseline", "first"),
            pruned=("ppl_pruned", "mean"),
            healed=("ppl_healed", "mean"),
            healed_std=("ppl_healed", "std"),
            recovered_pct=("recovered_pct", "mean"),
        )
        .round(2)
    )

final = summary.pivot_table(
    index=["model_short", "k"], columns="operator", values="mean"
).round(2)
final_std = summary.pivot_table(
    index=["model_short", "k"], columns="operator", values="std"
).round(2)
print("mean validation PPL")
display(final)
print("std over seeds")
display(final_std)

print("\ncompression and speed at k=1")
display(comp_df[comp_df.k == 1].round(3))

print("\npaired bridge-minus-removal difference (positive = bridge worse)")
display(paired_df.round(3))

print("\nrank-RANK_DEFAULT ceiling per block (the predictive diagnostic)")
display(diag.round(4))

## 15. Self-verification gate

Before exporting, this cell checks the run is internally consistent and prints a
single verdict. Read this output first: if any check fails, the numbers are not
usable and the cause is named.

Checks performed:

1. every cell parses and ran (implicit — you would not have reached this section)
2. the fit is bit-reproducible (Section 7)
3. every reported point has the full number of seeds
4. compression is monotonically non-increasing in $k$ for removal
5. the fitted bridge never claims more parameters than the block it replaced
6. the random control is disjoint from the BI targets
7. no NaN or infinite perplexity anywhere
8. the rank ceiling is a genuine upper bound on the achieved fit
9. the distillation term was non-trivial, so healing was not just fine-tuning
10. healing improved every pruned model it touched
11. the healed model still has the post-deletion parameter count
12. held-out test perplexity is consistent with validation for the healed model
13. BI removal never loses to random removal, which is the load-bearing half of F1

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -- {detail}" if detail else ""))


# 2) determinism
check(
    "fit is bit-reproducible",
    bool(det_df["deterministic"].all()),
    f"max |delta| = {det_df['abs_delta'].max():.2e}",
)

# 3) seed coverage
main_ops = ["baseline", "BI removal", "random removal", "fitted bridge"]
cov = results_df[results_df.operator.isin(main_ops)].groupby(
    ["operator", "model_short", "k"]
)["seed"].nunique()
expected = 1 if "baseline" in set(cov.index.get_level_values(0)) else len(SEEDS)
short_rows = cov[cov != expected]
check(
    "every point has the full seed count",
    len(short_rows) == 0,
    f"expected {expected}; offenders={len(short_rows)}",
)

# 4) compression monotone in k
mono_ok, mono_detail = True, []
for (short, op), s in comp_df.groupby(["model", "operator"]):
    s = s.sort_values("k")
    p = s["params_M"].to_numpy()
    if np.any(np.diff(p) > 1e-9):
        mono_ok = False
        mono_detail.append(f"{short}/{op}")
check(
    "parameter count is monotone in k",
    mono_ok,
    ", ".join(mono_detail) if mono_detail else "all operators",
)

# 5) bridge never inflates the model
bridge = results_df[results_df.operator == "fitted bridge"]
removal = results_df[results_df.operator == "BI removal"]
merged = bridge.merge(removal, on=["model_short", "k"], suffixes=("_br", "_rm"))
check(
    "bridge never exceeds plain removal in parameters",
    bool((merged["params_br"] <= merged["params_rm"] + 1).all()),
    f"max excess = {int((merged['params_br'] - merged['params_rm']).max())} params",
)

# 6) random control disjoint from BI targets
ctrl = results_df[results_df.operator == "random removal"]
overlaps = 0
for _, row in ctrl.iterrows():
    pick = row["target"]
    pick = ast.literal_eval(pick) if isinstance(pick, str) else list(pick)
    order = sorted(range(len(BI[row["model"]])), key=lambda i: BI[row["model"]][i])
    if set(pick) & set(order[: int(row["k"])]):
        overlaps += 1
check("random control disjoint from BI targets", overlaps == 0, f"{overlaps} overlaps")

# 7) no NaN / inf perplexity
vals = results_df["ppl"].to_numpy(dtype=float)
check(
    "all perplexities finite and positive",
    bool(np.all(np.isfinite(vals)) and np.all(vals > 0)),
    f"min={np.nanmin(vals):.3f} max={np.nanmax(vals):.4g}",
)

# 8) the rank ceiling really is an upper bound on the achieved local fit
ev_col = f"ev_{RANK_DEFAULT}"
ceiling_of = {}
for _, row in spectrum_df.iterrows():
    if not math.isnan(row.get(ev_col, np.nan)):
        # the ablation target is always the lowest-BI block of that model
        order = sorted(range(len(BI[row["model"]])), key=lambda i: BI[row["model"]][i])
        if row["layer"] == order[0]:
            ceiling_of[row["model_short"]] = float(row[ev_col])

bad_bound = []
for _, row in ablation_df[ablation_df.knob == "rank"].iterrows():
    ceil = ceiling_of.get(row["model_short"])
    if ceil is not None and row["fit_ev"] > ceil + 1e-6:
        bad_bound.append(
            f"{row['model_short']} rank={row['value']}: "
            f"fit {row['fit_ev']:.3f} > ceiling {ceil:.3f}"
        )
check(
    "achieved fit never exceeds the rank ceiling",
    len(bad_bound) == 0,
    "; ".join(bad_bound[:3]) if bad_bound else "consistent with Eckart-Young",
)

# F1's load-bearing claim, checked rather than asserted: the BI ranking must
# never actually hurt. Magnitude is allowed to tie (it does, at SmolLM k=1);
# losing is not.
bi_worse = reliability_df[reliability_df["bi_mean"] > reliability_df["random_mean"]]
check(
    "BI removal is never worse than random removal (F1)",
    len(bi_worse) == 0,
    f"{len(bi_worse)} of {len(reliability_df)} (model, k) points lost to random"
    + ("; " + "; ".join(
        f"{r.model} k={r.k}: {r.bi_mean:.2f} vs {r.random_mean:.2f}"
        for r in bi_worse.itertuples()
    ) if len(bi_worse) else ""),
)

# ---- healing-specific checks ------------------------------------------------
if len(heal_df):
    check(
        "distillation term was non-trivial",
        bool(heal_df["kl_first"].mean() > 1e-3),
        f"mean initial KL = {heal_df['kl_first'].mean():.4f} "
        f"(0 would mean this was fine-tuning, not distilling)",
    )
    regressed = heal_df[heal_df["ppl_healed"] >= heal_df["ppl_pruned"]]
    check(
        "healing improved every pruned model it touched",
        len(regressed) == 0,
        f"{len(regressed)} of {len(heal_df)} runs did not improve",
    )
    # Healing must not quietly put the deleted parameters back. Parameter count
    # depends only on the target set, so one rebuild per distinct (model, target)
    # covers every seed -- otherwise this re-loads a model 40 times at SMALL=False.
    hp = []
    cache = {}
    for r in heal_df.itertuples():
        key = (r.model_id, r.target)
        if key not in cache:
            m = hard_remove_fp32(r.model_id, ast.literal_eval(r.target))
            cache[key] = n_params(m)
            free(m)
        if cache[key] != r.params:
            hp.append(f"{r.model_short} k={r.k}: {cache[key]} != {r.params}")
    check(
        "healed model keeps the post-deletion parameter count",
        len(hp) == 0,
        "; ".join(hp[:3]) if hp else "no parameters resurrected by training",
    )
    # Validation and test must not diverge wildly, or one of them leaked.
    drift = (
        (heal_df["ppl_healed"] - heal_df["ppl_test"]).abs()
        / heal_df["ppl_healed"]
    )
    check(
        "held-out test PPL is consistent with validation",
        bool((drift < 0.35).all()),
        f"max relative drift = {drift.max():.1%}",
    )
else:
    print("\n[SKIP] healing checks: no healing runs recorded "
          "(every model exceeded HEAL_MAX_PARAMS)")

verdict_df = pd.DataFrame(checks)
display(verdict_df)
n_pass = int(verdict_df["passed"].sum())
print(f"\n{n_pass}/{len(verdict_df)} checks passed.")

if n_pass == len(verdict_df):
    print("VERDICT: run is internally consistent. The numbers in Section 14 stand.")
else:
    print(
        "VERDICT: at least one check failed. Do not quote these numbers until the "
        "failing check above is understood."
    )

## 16. Export and download

Everything is written to a `results/` folder next to the notebook. On Colab that
folder lives in the runtime and is **deleted when the session ends**, so the last
cell bundles it into a single zip and triggers a browser download. Save that zip;
it is the evidence.

In [ ]:
import os

os.makedirs("results", exist_ok=True)

results_df.to_csv("results/main_comparison.csv", index=False)
comp_df.to_csv("results/compression.csv", index=False)
paired_df.to_csv("results/paired_comparison.csv", index=False)
reliability_df.to_csv("results/bi_vs_random.csv", index=False)
spectrum_df.to_csv("results/residual_spectrum.csv", index=False)
ablation_df.to_csv("results/ablations.csv", index=False)
corr_df.to_csv("results/rank_correlation.csv", index=False)
test_df.to_csv("results/test_split.csv", index=False)
det_df.to_csv("results/determinism.csv", index=False)
if len(heal_df):
    heal_df.to_csv("results/healing.csv", index=False)
if heal_skipped:
    pd.DataFrame(heal_skipped).to_csv("results/healing_skipped.csv", index=False)
try:
    guided_df.to_csv("results/ev_guided.csv", index=False)
except NameError:
    pass

meta = {
    "device": device,
    "device_banner": device_banner(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "model_dtype": str(MODEL_DTYPE),
    "dataset": f"Salesforce/wikitext / {WIKITEXT_CONFIG}",
    "eval_tokens": EVAL_TOKENS,
    "eval_block": EVAL_BLOCK,
    "calib_chars": CALIB_CHARS,
    "seeds": list(SEEDS),
    "SMALL": SMALL,
    "rank": RANK_DEFAULT,
    "bridge_bits": BRIDGE_BITS,
    "models": MODEL_IDS,
    "deterministic": bool(det_df["deterministic"].all()),
    "healing": {
        "steps": HEAL_STEPS,
        "lr": HEAL_LR,
        "temperature": HEAL_TEMP,
        "alpha": HEAL_ALPHA,
        "ks": list(HEAL_KS),
        "max_params": HEAL_MAX_PARAMS,
        "runs": int(len(heal_df)),
        "skipped": heal_skipped,
    },
    "checks_passed": int(verdict_df["passed"].sum()),
    "checks_total": int(len(verdict_df)),
}
with open("results/meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

print("wrote:")
for f in sorted(os.listdir("results")):
    print("   results/" + f)

# ---- bundle everything into one downloadable archive ---------------------
import shutil

stamp = meta["device"].replace("/", "_")
archive = shutil.make_archive(
    f"evidence_{stamp}_SMALL{int(SMALL)}", "zip", root_dir="results"
)
size_mb = os.path.getsize(archive) / 1e6
print(f"\nbundled -> {archive}  ({size_mb:.2f} MB)")

# Works on Colab (saves to /content and pops a download dialog) and is a no-op
# elsewhere. Try both trigger styles so it fires regardless of Colab version.
try:
    from google.colab import files

    files.download(archive)
    print("download started via google.colab.files")
except Exception:
    try:
        from IPython.display import display, HTML

        display(
            HTML(
                f'<a download="{archive}" href="sandbox:/content/{archive}">'
                f"Download {archive}</a>"
            )
        )
    except Exception:
        print(f"could not trigger a download; fetch {archive} manually")

print(
    "\nEvidence checklist -- keep these with the report:\n"
    "  results/meta.json               device, versions, seeds, determinism flag\n"
    "  results/determinism.csv         bit-reproducibility proof\n"
    "  results/main_comparison.csv     every PPL/params/latency run, per seed\n"
    "  results/compression.csv         measured parameter and latency deltas\n"
    "  results/paired_comparison.csv   bridge-minus-removal, seed by seed\n"
    "  results/residual_spectrum.csv   training-free rank ceilings per block\n"
    "  results/ablations.csv           rank / bit-width / calibration sweeps\n"
    "  results/rank_correlation.csv    the local-fit vs perplexity anti-correlation\n"
    "  results/ev_guided.csv           EV-guided vs BI-guided selection\n"
    "  results/test_split.csv          held-out test perplexity\n"
    "  evidence_*.zip                  all of the above, plus the run's config\n"
)
print()
print(json.dumps(meta, indent=2))